https://github.com/codercap1/Tri-Tue-Nhan-Tao-nhom07.git

Nguyễn Văn Trung - 24110365

In [2]:
import tkinter as tk
from tkinter import ttk, scrolledtext
import random
import heapq
import time
import threading
from collections import deque
import math as _math
# ══════════════════════════════════════════════════════════
#  HẰNG SỐ & TIỆN ÍCH
# ══════════════════════════════════════════════════════════

GOAL_STATE = (1, 2, 3, 4, 5, 6, 7, 8, 0)
# Vị trí đích của mỗi ô theo GOAL_STATE = (1,2,3,4,5,6,7,8,0)
_GOAL_POS = {GOAL_STATE[i]: divmod(i, 3) for i in range(9)}

# 4 hướng di chuyển (dr, dc, tên hành động)
DIRECTIONS = [(-1, 0, "Up"), (1, 0, "Down"), (0, -1, "Left"), (0, 1, "Right")]

#manhattan
def h_manhattan(state: tuple) -> int:
    """h(n) – Tổng khoảng cách Manhattan mỗi ô về đích (bỏ qua ô 0)."""
    total = 0
    for i, val in enumerate(state):
        if val != 0:
            r, c = divmod(i, 3)
            gr, gc = _GOAL_POS[val]
            total += abs(r - gr) + abs(c - gc)
    return total

#đường chim bay
def h_euclidean(state: tuple) -> float:
    """h(n) – Tổng khoảng cách Euclidean (đường chim bay) mỗi ô về đích (bỏ qua ô 0)."""
    total = 0.0
    for i, val in enumerate(state):
        if val != 0:
            r, c = divmod(i, 3)
            gr, gc = _GOAL_POS[val]
            total += _math.sqrt((r - gr) ** 2 + (c - gc) ** 2)
    return total

#số dãy ngược
def g_inversions(state: tuple) -> int:
    """g(n) – Số dãy ngược (inversions) trong state, không tính ô 0."""
    flat = [v for v in state if v != 0]
    inv  = sum(1 for i in range(len(flat))
                 for j in range(i + 1, len(flat))
                 if flat[i] > flat[j])
    return inv

def is_solvable(state: tuple) -> bool:
    """Kiểm tra trạng thái có giải được không (đếm nghịch thế – inversion count)."""
    flat = [v for v in state if v != 0]
    inv  = sum(1 for i in range(len(flat))
                 for j in range(i + 1, len(flat))
                 if flat[i] > flat[j])
    return inv % 2 == 0


def random_solvable_state() -> tuple:
    """Sinh ngẫu nhiên trạng thái chắc chắn giải được."""
    s = list(range(9))
    while True:
        random.shuffle(s)
        if is_solvable(s):
            return tuple(s)


def expand(state: tuple) -> list:
    """Trả về [(new_state, action), …] – tất cả nước đi hợp lệ từ state."""
    zero    = state.index(0)
    row, col = divmod(zero, 3)
    result  = []
    for dr, dc, action in DIRECTIONS:
        nr, nc = row + dr, col + dc
        if 0 <= nr < 3 and 0 <= nc < 3:
            lst          = list(state)
            ni           = nr * 3 + nc
            lst[zero], lst[ni] = lst[ni], lst[zero]
            result.append((tuple(lst), action))
    return result



class Node:
    """
    Node = {
        state     : current state,
        parent    : reference to parent node,
        action    : action from parent to reach this node,
        path_cost : total cost from root
    }
    """
    __slots__ = ("state", "parent", "action", "path_cost", "depth")

    def __init__(self, state, parent=None, action=None, path_cost=0):
        self.state     = state
        self.parent    = parent
        self.action    = action
        self.path_cost = path_cost
        self.depth     = 0 if parent is None else parent.depth + 1

    # Truy vết lời giải từ node về root
    def solution(self) -> list:
        """Trả về [(state, action), …] từ bước đầu đến bước hiện tại."""
        path, node = [], self
        while node.parent is not None:
            path.append((node.state, node.action))
            node = node.parent
        path.reverse()
        return path

    # IS-CYCLE: kiểm tra state có lặp trong đường đi hiện tại không (dùng cho IDS)
    def is_cycle(self) -> bool:
        ancestor = self.parent
        while ancestor is not None:
            if ancestor.state == self.state:
                return True
            ancestor = ancestor.parent
        return False

    # Để heapq so sánh được (UCS)
    def __lt__(self, other):
        return self.path_cost < other.path_cost


# ══════════════════════════════════════════════════════════
#  THUẬT TOÁN  
# ══════════════════════════════════════════════════════════

# ══════════════════════════════════════════════════════════
#  THUẬT TOÁN – GREEDY BEST-FIRST SEARCH
#  h(n) : khoảng cách Manhattan
# ══════════════════════════════════════════════════════════

def greedy(start: tuple, goal: tuple, log_cb) -> list | None:
    """
    Greedy Best-First Search
    ─────────────────────────
    Ưu tiên trạng thái có h(n) nhỏ nhất (Manhattan distance).
    Không xét lại trạng thái đã có trong FRONTIER hoặc REACHED.
    """
    node = Node(start)
    if node.state == goal:
        log_cb("Greedy: trạng thái ban đầu đã là đích!", "ok")
        return []

    counter        = 0
    h0             = h_manhattan(start)
    frontier       = [(h0, counter, node)]   # min-heap theo h(n)
    reached        = {start}                 # FRONTIER ∪ EXPLORED
    explored       = set()
    expanded_count = 0

    while frontier:
        h, _, node = heapq.heappop(frontier)

        if node.state in explored:
            continue

        if node.state == goal:
            log_cb(
                f"Greedy ✓  |  Nodes expanded: {expanded_count}"
                f"  |  Steps: {node.depth}"
                f"  |  h(Start)={h_manhattan(start)}",
                "ok"
            )
            return node.solution()

        explored.add(node.state)
        expanded_count += 1

        for child_state, action in expand(node.state):
            if child_state not in reached:       # chưa có trong FRONTIER & REACHED
                child = Node(child_state, node, action, node.path_cost + 1)
                hn    = h_manhattan(child_state)
                counter += 1
                heapq.heappush(frontier, (hn, counter, child))
                reached.add(child_state)
            # đã có → bỏ qua (theo slide)

    log_cb("Greedy: không tìm được lời giải!", "warn")
    return None

# ══════════════════════════════════════════════════════════
#  THUẬT TOÁN – A*
#  g(n) : số dãy ngược (inversions, không tính ô 0)
#  h(n) : khoảng cách Euclidean (đường chim bay)
#  f(n) = g(n) + h(n)
# ══════════════════════════════════════════════════════════

def a_star(start: tuple, goal: tuple, log_cb) -> list | None:
    """
    A* Search
    ──────────
    f(n) = g(n) + h(n)
      g(n) = số dãy ngược của trạng thái n (không tính ô 0)
      h(n) = tổng khoảng cách Euclidean mỗi ô về đích
    Mở lại trạng thái nếu tìm được f(n) tốt hơn.
    """
    node = Node(start)
    if node.state == goal:
        log_cb("A*: trạng thái ban đầu đã là đích!", "ok")
        return []

    g0 = g_inversions(start)
    h0 = h_euclidean(start)
    f0 = g0 + h0

    counter        = 0
    frontier       = [(f0, counter, node)]   # min-heap theo f(n)
    best_f         = {start: f0}             # state → f tốt nhất đã biết
    explored       = set()
    expanded_count = 0

    while frontier:
        f, _, node = heapq.heappop(frontier)

        if node.state in explored:
            continue

        if node.state == goal:
            log_cb(
                f"A* ✓  |  Nodes expanded: {expanded_count}"
                f"  |  Steps: {node.depth}"
                f"  |  f(goal)={f:.3f}"
                f"  |  g(start)={g_inversions(start)}"
                f"  |  h(start)={h_euclidean(start):.3f}",
                "ok"
            )
            return node.solution()

        explored.add(node.state)
        expanded_count += 1

        for child_state, action in expand(node.state):
            if child_state not in explored:
                gm = g_inversions(child_state)
                hm = h_euclidean(child_state)
                fm = gm + hm

                if child_state in best_f and fm >= best_f[child_state]:
                    continue             # tệ hơn hoặc bằng → bỏ qua

                # Cập nhật hoặc thêm mới vào FRONTIER
                best_f[child_state] = fm
                child   = Node(child_state, node, action, gm)
                counter += 1
                heapq.heappush(frontier, (fm, counter, child))

    log_cb("A*: không tìm được lời giải!", "warn")
    return None

def bfs(start: tuple, goal: tuple, log_cb) -> list | None:
    """
    BFS – Graph Search  (Image 2)
    ─────────────────────────────
    frontier ← FIFO-QUEUE
    Kiểm tra goal khi INSERT vào frontier (early goal test)
    """
    node = Node(start)
    if node.state == goal:
        log_cb("BFS: trạng thái ban đầu đã là đích!", "ok")
        return []

    frontier       = deque([node])      # FIFO-QUEUE
    frontier_set   = {start}            # để kiểm tra child ∉ frontier O(1)
    explored       = set()              # tập đã khám phá
    expanded_count = 0

    while frontier:
        node = frontier.popleft()       # REMOVE() – lấy node đầu tiên trong queue
        explored.add(node.state)
        frontier_set.discard(node.state)
        expanded_count += 1

        for child_state, action in expand(node.state):
            # child.STATE ∉ explored ∧ child ∉ frontier
            if child_state not in explored and child_state not in frontier_set:
                child = Node(child_state, node, action, node.path_cost + 1)

                if child.state == goal:     # GOAL-TEST
                    log_cb(f"BFS ✓  |  Nodes expanded: {expanded_count}  |  "
                           f"Steps: {child.depth}", "ok")
                    return child.solution()

                frontier.append(child)      # frontier.INSERT(child)
                frontier_set.add(child_state)

    log_cb("BFS: không tìm được lời giải!", "warn")
    return None


def dfs(start: tuple, goal: tuple, log_cb, depth_limit: int = 150) -> list | None:
    """
    DFS – Graph Search  (Image 4)
    ─────────────────────────────
    frontier ← LIFO-STACK  (thay FIFO → LIFO so với BFS)
    depth_limit: ngăn DFS lún quá sâu (mặc định 150)
    """
    node = Node(start)
    if node.state == goal:
        log_cb("DFS: trạng thái ban đầu đã là đích!", "ok")
        return []

    frontier       = [node]             # LIFO-STACK
    frontier_set   = {start}
    explored       = set()
    expanded_count = 0

    while frontier:
        node = frontier.pop()           # REMOVE() – lấy từ đỉnh stack

        if node.state in explored:
            continue

        explored.add(node.state)
        frontier_set.discard(node.state)
        expanded_count += 1

        if node.state == goal:
            log_cb(f"DFS ✓  |  Nodes expanded: {expanded_count}  |  "
                   f"Steps: {node.depth}  |  depth_limit={depth_limit}", "ok")
            return node.solution()

        if node.depth >= depth_limit:
            continue                    # cắt nhánh quá sâu

        for child_state, action in expand(node.state):
            if child_state not in explored and child_state not in frontier_set:
                child = Node(child_state, node, action, node.path_cost + 1)
                frontier.append(child)
                frontier_set.add(child_state)

    log_cb(f"DFS: không tìm được lời giải (depth_limit={depth_limit})!", "warn")
    return None


def ids(start: tuple, goal: tuple, log_cb) -> list | None:
    """
    IDS – Iterative Deepening Search  (Image 5)
    ─────────────────────────────────────────────
    Gọi DEPTH-LIMITED-SEARCH với depth tăng dần từ 0 → ∞
    """
    total_expanded = [0]

    def depth_limited_search(depth_limit: int):
        """
        DEPTH-LIMITED-SEARCH(problem, l)
        frontier ← LIFO queue (stack) with NODE(problem.INITIAL)
        result ← failure
        """
        root_node = Node(start)
        frontier  = [root_node]         # LIFO stack
        result    = None                # result ← failure

        while frontier:                 # while not IS-EMPTY(frontier) do
            node = frontier.pop()       # node ← POP(frontier)
            total_expanded[0] += 1

            if node.state == goal:      # if problem.IsGoal(node.STATE) then return node
                return node

            if node.depth > depth_limit:    # if DEPTH(node) > l then
                result = "cutoff"           #   result ← cutoff
            elif not node.is_cycle():       # else if not IS-CYCLE(node) do
                for child_state, action in expand(node.state):
                    child = Node(child_state, node, action, node.path_cost + 1)
                    frontier.append(child)  # add child to frontier

        return result                   # return result

    # for depth = 0 to ∞ do
    for depth in range(0, 300):
        log_cb(f"IDS: thử depth limit = {depth} …", "step")
        result = depth_limited_search(depth)

        if result == "cutoff":
            continue                    # result = cutoff → thử depth lớn hơn
        elif result is None:
            log_cb("IDS: không tìm được lời giải!", "warn")
            return None
        else:
            path = result.solution()
            log_cb(f"IDS ✓  |  Depth tìm thấy: {depth}  |  "
                   f"Total nodes expanded: {total_expanded[0]}  |  "
                   f"Steps: {len(path)}", "ok")
            return path

    log_cb("IDS: vượt quá giới hạn depth!", "warn")
    return None


def ucs(start: tuple, goal: tuple, log_cb) -> list | None:
    """
    UCS – Uniform Cost Search
    ──────────────────────────
    Cấu trúc giống BFS nhưng frontier là priority queue theo path_cost.
    Với 8-puzzle (cost mỗi bước = 1) kết quả tương đương BFS.
    """
    node = Node(start)
    if node.state == goal:
        log_cb("UCS: trạng thái ban đầu đã là đích!", "ok")
        return []

    counter        = 0                  # tie-breaker để heapq không so sánh Node
    frontier       = [(0, counter, node)]
    best_cost      = {start: 0}        # state → chi phí tốt nhất đã biết
    explored       = set()
    expanded_count = 0

    while frontier:
        cost, _, node = heapq.heappop(frontier)

        if node.state in explored:
            continue

        if node.state == goal:
            log_cb(f"UCS ✓  |  Nodes expanded: {expanded_count}  |  "
                   f"Steps: {node.depth}  |  Total cost: {cost}", "ok")
            return node.solution()

        explored.add(node.state)
        expanded_count += 1

        for child_state, action in expand(node.state):
            child_cost = cost + 1       # step cost = 1
            if child_state not in explored:
                if child_state not in best_cost or child_cost < best_cost[child_state]:
                    best_cost[child_state] = child_cost
                    child   = Node(child_state, node, action, child_cost)
                    counter += 1
                    heapq.heappush(frontier, (child_cost, counter, child))

    log_cb("UCS: không tìm được lời giải!", "warn")
    return None


# ══════════════════════════════════════════════════════════
#  ALGORITHM REGISTRY
#  ← THÊM THUẬT TOÁN MỚI Ở ĐÂY ←
#
#  Chữ ký bắt buộc:
#    def my_algo(start: tuple, goal: tuple, log_cb) -> list | None
#      log_cb(msg, tag)  –  tag: "info" | "ok" | "warn" | "step"
#      return list[(state, action)]  hoặc  None
# ══════════════════════════════════════════════════════════

ALGORITHMS: dict = {
    "BFS": bfs,
    "DFS": dfs,
    "IDS": ids,
    "UCS": ucs,
    "Greedy": greedy,     # Greedy Best-First – h(n)=Manhattan
    "A*"    : a_star,     # A* – g(n)=inversions, h(n)=Euclidean
}


# ══════════════════════════════════════════════════════════
#  GUI CONSTANTS
# ══════════════════════════════════════════════════════════

CELL_MAIN  = 82      # px mỗi ô trong Box3
CELL_SMALL = 40      # px mỗi ô trong Box2

C_TILE   = "#2563EB"    # màu ô số
C_EMPTY  = "#E5E7EB"    # màu ô trống
C_BG     = "#D1D5DB"    # nền tổng
C_TEXT   = "white"


# ══════════════════════════════════════════════════════════
#  APP CLASS
# ══════════════════════════════════════════════════════════

class PuzzleApp:
    def __init__(self, root: tk.Tk):
        self.root = root
        self.root.title("8-Puzzle Solver")
        self.root.configure(bg=C_BG)
        self.root.resizable(True, True)

        # ── trạng thái ──
        self.goal_state    = GOAL_STATE
        self.initial_state = random_solvable_state()
        self.current_state = self.initial_state
        self.solution_path: list = []
        self.current_step  = 0
        self.is_running    = False
        self.is_paused     = False
        self.anim_job      = None
        self.speed_ms      = 350

        self._build_ui()
        self._refresh_all()
        self._log("Chào mừng! Nhấn [🔀 Random] để tạo puzzle, [▶ Start] để giải.", "info")

    # ─────────────────────────────────────────
    #  XÂY DỰNG GIAO DIỆN
    # ─────────────────────────────────────────

    def _build_ui(self):
        root = self.root
        PAD  = 8

        outer = tk.Frame(root, bg=C_BG, padx=PAD, pady=PAD)
        outer.pack(fill=tk.BOTH, expand=True)

        # ── LEFT PANEL ──────────────────────
        left = tk.Frame(outer, bg=C_BG)
        left.pack(side=tk.LEFT, fill=tk.Y, padx=(0, PAD))

        # ── BOX 1 · Dropdown thuật toán ────
        box1 = tk.LabelFrame(left, text=" Ô 1 · Thuật toán ",
                             bg=C_BG, font=("Arial", 9, "bold"),
                             fg="#1F2937", padx=8, pady=6)
        box1.pack(fill=tk.X, pady=(0, 6))

        self.algo_var   = tk.StringVar(value=list(ALGORITHMS)[0])
        self.algo_combo = ttk.Combobox(
            box1, textvariable=self.algo_var,
            values=list(ALGORITHMS.keys()),
            state="readonly", width=20, font=("Arial", 11)
        )
        self.algo_combo.pack(pady=2)
        self.algo_combo.bind("<<ComboboxSelected>>", self._on_algo_change)

        # ── BOX 3 · Animation ───────────────
        box3 = tk.LabelFrame(left, text=" Ô 3 · Trạng thái hiện tại ",
                             bg=C_BG, font=("Arial", 9, "bold"),
                             fg="#1F2937", padx=6, pady=6)
        box3.pack(fill=tk.X, pady=(0, 6))

        w = h = CELL_MAIN * 3
        self.main_canvas = tk.Canvas(box3, width=w, height=h,
                                     bg="white", highlightthickness=1,
                                     highlightbackground="#9CA3AF")
        self.main_canvas.pack()

        self.step_var = tk.StringVar(value="Bước: 0 / 0")
        tk.Label(box3, textvariable=self.step_var, bg=C_BG,
                 font=("Consolas", 9), fg="#374151").pack(pady=(4, 0))

        # ── BOX 5 · Buttons + Speed ─────────
        box5 = tk.LabelFrame(left, text=" Ô 5 · Điều khiển ",
                             bg=C_BG, font=("Arial", 9, "bold"),
                             fg="#1F2937", padx=6, pady=6)
        box5.pack(fill=tk.X, pady=(0, 6))

        btn_row = tk.Frame(box5, bg=C_BG)
        btn_row.pack()
        btn_kw = dict(font=("Arial", 10, "bold"), relief=tk.FLAT,
                      cursor="hand2", padx=8, pady=5)

        tk.Button(btn_row, text="🔀 Random", bg="#059669", fg="white",
                  command=self._randomize, **btn_kw).grid(row=0, column=0, padx=3, pady=3)

        self.start_btn = tk.Button(btn_row, text="▶ Start", bg="#2563EB", fg="white",
                                   command=self._start, **btn_kw)
        self.start_btn.grid(row=0, column=1, padx=3, pady=3)

        self.pause_btn = tk.Button(btn_row, text="⏸ Pause", bg="#D97706", fg="white",
                                   command=self._toggle_pause,
                                   state=tk.DISABLED, **btn_kw)
        self.pause_btn.grid(row=0, column=2, padx=3, pady=3)

        # Speed row
        spd = tk.Frame(box5, bg=C_BG)
        spd.pack(fill=tk.X, pady=(6, 0))
        tk.Label(spd, text="⚡ Speed:", bg=C_BG, font=("Arial", 9)).pack(side=tk.LEFT, padx=(2, 4))
        self.speed_var = tk.IntVar(value=self.speed_ms)
        ttk.Scale(spd, from_=30, to=1500, variable=self.speed_var,
                  orient=tk.HORIZONTAL, command=self._on_speed
                  ).pack(side=tk.LEFT, fill=tk.X, expand=True)
        self.spd_lbl = tk.Label(spd, text=f"{self.speed_ms}ms",
                                bg=C_BG, font=("Consolas", 9), width=7)
        self.spd_lbl.pack(side=tk.LEFT)

        # ── BOX 2 · Initial & Goal ──────────
        box2 = tk.LabelFrame(left, text=" Ô 2 · Trạng thái ",
                             bg=C_BG, font=("Arial", 9, "bold"),
                             fg="#1F2937", padx=6, pady=6)
        box2.pack(fill=tk.X)

        row2 = tk.Frame(box2, bg=C_BG)
        row2.pack()

        # 2.1 Ban đầu
        f21 = tk.LabelFrame(row2, text="2.1 Ban đầu", bg=C_BG,
                            font=("Arial", 8), fg="#374151")
        f21.grid(row=0, column=0, padx=4)
        self.init_canvas = tk.Canvas(f21,
                                     width=CELL_SMALL * 3, height=CELL_SMALL * 3,
                                     bg="white", highlightthickness=0)
        self.init_canvas.pack(padx=2, pady=2)

        # Mũi tên "2 →"
        mid = tk.Frame(row2, bg=C_BG)
        mid.grid(row=0, column=1, padx=8)
        tk.Label(mid, text="2", bg=C_BG, font=("Arial", 8), fg="#6B7280").pack()
        tk.Label(mid, text="→", bg=C_BG, font=("Arial", 20, "bold"), fg="#1F2937").pack()

        # 2.2 Đích
        f22 = tk.LabelFrame(row2, text="2.2 Đích", bg=C_BG,
                            font=("Arial", 8), fg="#374151")
        f22.grid(row=0, column=2, padx=4)
        self.goal_canvas = tk.Canvas(f22,
                                     width=CELL_SMALL * 3, height=CELL_SMALL * 3,
                                     bg="white", highlightthickness=0)
        self.goal_canvas.pack(padx=2, pady=2)

        # ── RIGHT PANEL · BOX 4 Log ─────────
        box4 = tk.LabelFrame(outer, text=" Ô 4 · Log thực hiện ",
                             bg=C_BG, font=("Arial", 9, "bold"),
                             fg="#1F2937", padx=6, pady=6)
        box4.pack(side=tk.LEFT, fill=tk.BOTH, expand=True)

        self.log_box = scrolledtext.ScrolledText(
            box4, state=tk.DISABLED, wrap=tk.WORD,
            font=("Consolas", 9), bg="#0F172A", fg="#CBD5E1",
            insertbackground="white", relief=tk.FLAT
        )
        self.log_box.pack(fill=tk.BOTH, expand=True)

        # Màu log theo tag
        self.log_box.tag_config("info", foreground="#93C5FD")   # xanh nhạt
        self.log_box.tag_config("ok",   foreground="#6EE7B7")   # xanh lá
        self.log_box.tag_config("warn", foreground="#FCD34D")   # vàng
        self.log_box.tag_config("step", foreground="#94A3B8")   # xám

        tk.Button(box4, text="🗑  Xóa log", bg="#374151", fg="white",
                  font=("Arial", 9), relief=tk.FLAT, cursor="hand2",
                  command=self._clear_log).pack(pady=(6, 0))

    # ─────────────────────────────────────────
    #  VẼ PUZZLE
    # ─────────────────────────────────────────

    def _draw_puzzle(self, canvas: tk.Canvas, state: tuple, cell: int):
        canvas.delete("all")
        for i, val in enumerate(state):
            r, c = divmod(i, 3)
            x1   = c * cell + 2
            y1   = r * cell + 2
            x2   = x1 + cell - 4
            y2   = y1 + cell - 4
            if val == 0:
                canvas.create_rectangle(x1, y1, x2, y2,
                                        fill=C_EMPTY, outline="#9CA3AF", width=2)
            else:
                canvas.create_rectangle(x1, y1, x2, y2,
                                        fill=C_TILE, outline="#1D4ED8", width=2)
                fs = max(cell // 3, 10)
                canvas.create_text((x1 + x2) // 2, (y1 + y2) // 2,
                                   text=str(val), fill=C_TEXT,
                                   font=("Arial", fs, "bold"))

    def _refresh_all(self):
        self._draw_puzzle(self.main_canvas,  self.current_state,  CELL_MAIN)
        self._draw_puzzle(self.init_canvas,  self.initial_state,  CELL_SMALL)
        self._draw_puzzle(self.goal_canvas,  self.goal_state,     CELL_SMALL)

    # ─────────────────────────────────────────
    #  LOGGING
    # ─────────────────────────────────────────

    def _log(self, msg: str, tag: str = "info"):
        ts = time.strftime("%H:%M:%S")
        self.log_box.config(state=tk.NORMAL)
        self.log_box.insert(tk.END, f"[{ts}]  {msg}\n", tag)
        self.log_box.see(tk.END)
        self.log_box.config(state=tk.DISABLED)

    def _log_safe(self, msg: str, tag: str = "info"):
        """Thread-safe: schedule log trên main thread."""
        self.root.after(0, lambda m=msg, t=tag: self._log(m, t))

    def _clear_log(self):
        self.log_box.config(state=tk.NORMAL)
        self.log_box.delete("1.0", tk.END)
        self.log_box.config(state=tk.DISABLED)

    # ─────────────────────────────────────────
    #  EVENT HANDLERS
    # ─────────────────────────────────────────

    def _randomize(self):
        self._stop()
        self.initial_state = random_solvable_state()
        self.current_state = self.initial_state
        self.solution_path = []
        self.current_step  = 0
        self.step_var.set("Bước: 0 / 0")
        self._refresh_all()
        self._log(f"Puzzle mới: {list(self.initial_state)}", "warn")

    def _on_algo_change(self, _=None):
        self._stop()
        self.current_state = self.initial_state
        self.current_step  = 0
        self.step_var.set("Bước: 0 / 0")
        self._draw_puzzle(self.main_canvas, self.current_state, CELL_MAIN)
        self._log(f"Đã chọn thuật toán: {self.algo_var.get()}", "warn")

    def _on_speed(self, val):
        self.speed_ms = int(float(val))
        self.spd_lbl.config(text=f"{self.speed_ms}ms")

    # ─────────────────────────────────────────
    #  SOLVE & ANIMATION
    # ─────────────────────────────────────────

    def _start(self):
        if self.is_running:
            return

        name = self.algo_var.get()
        fn   = ALGORITHMS.get(name)
        if not fn:
            self._log(f"Thuật toán '{name}' chưa được đăng ký!", "warn")
            return

        # reset về trạng thái ban đầu
        self.current_state = self.initial_state
        self.current_step  = 0
        self._draw_puzzle(self.main_canvas, self.current_state, CELL_MAIN)
        self.step_var.set("Bước: 0 / 0")

        self._log(f"{'═'*42}", "info")
        self._log(f"Thuật toán : {name}", "info")
        self._log(f"Start      : {list(self.initial_state)}", "info")
        self._log(f"Goal       : {list(self.goal_state)}", "info")
        self._log(f"Đang tìm lời giải …", "warn")

        self.start_btn.config(state=tk.DISABLED)
        self.pause_btn.config(state=tk.NORMAL, text="⏸ Pause")

        def worker():
            t0   = time.perf_counter()
            path = fn(self.initial_state, self.goal_state, self._log_safe)
            dt   = time.perf_counter() - t0
            if path is not None:
                self._log_safe(f"Thời gian tìm kiếm : {dt:.4f}s", "ok")
                self.solution_path = path
                self.root.after(0, self._start_animation)
            else:
                self._log_safe("Không tìm được lời giải!", "warn")
                self.root.after(0, self._reset_buttons)

        threading.Thread(target=worker, daemon=True).start()

    def _start_animation(self):
        self.is_running   = True
        self.is_paused    = False
        self.current_step = 0
        total = len(self.solution_path)
        self.step_var.set(f"Bước: 0 / {total}")
        self._tick()

    def _tick(self):
        """Một bước animation."""
        if not self.is_running or self.is_paused:
            return

        total = len(self.solution_path)
        if self.current_step < total:
            state, action = self.solution_path[self.current_step]
            self.current_state = state
            self.current_step += 1
            self.step_var.set(f"Bước: {self.current_step} / {total}")
            self._draw_puzzle(self.main_canvas, state, CELL_MAIN)
            self._log(
                f"  Bước {self.current_step:>3}/{total}  "
                f"[{action:<5}]  {list(state)}",
                "step"
            )
            self.anim_job = self.root.after(self.speed_ms, self._tick)
        else:
            self.is_running = False
            self._log("✅  Puzzle đã được giải xong!", "ok")
            self._log(f"{'═'*42}", "info")
            self._reset_buttons()

    def _toggle_pause(self):
        if not self.is_running:
            return
        self.is_paused = not self.is_paused
        if self.is_paused:
            if self.anim_job:
                self.root.after_cancel(self.anim_job)
            self.pause_btn.config(text="▶ Tiếp")
            self._log("⏸  Tạm dừng.", "warn")
        else:
            self.pause_btn.config(text="⏸ Pause")
            self._log("▶  Tiếp tục.", "warn")
            self._tick()

    def _stop(self):
        self.is_running = False
        self.is_paused  = False
        if self.anim_job:
            self.root.after_cancel(self.anim_job)
            self.anim_job = None
        self._reset_buttons()

    def _reset_buttons(self):
        self.start_btn.config(state=tk.NORMAL)
        self.pause_btn.config(state=tk.DISABLED, text="⏸ Pause")


# ══════════════════════════════════════════════════════════
#  ENTRY POINT
# ══════════════════════════════════════════════════════════

if __name__ == "__main__":
    root = tk.Tk()
    root.minsize(720, 640)
    PuzzleApp(root)
    root.mainloop()